In [1]:
import pandas as pd
import numpy as np

# 1. Load original merged raw dataset
input_file = 'cleaned_allmerged_aqi_weather.csv'
df = pd.read_csv(input_file)

# 2. Filter for Valid Quality Control records
df_valid = df[df['QC Name'] == 'Valid'].copy()

# 3. Parse DateTime and sort chronologically
df_valid['Date (LT)'] = pd.to_datetime(df_valid['Date (LT)'])
df_valid = df_valid.sort_values('Date (LT)').reset_index(drop=True)

# 4. Define Bangladesh Climatological Seasons & Cross-Year Winter Grouping
def get_season_info(dt):
    month = dt.month
    year = dt.year

    if month in [12, 1, 2]:
        season = 'Winter'
        # Group Dec of Year N with Jan/Feb of Year N+1 as "Winter_N-N+1"
        season_year = f"Winter_{year-1}-{year}" if month in [1, 2] else f"Winter_{year}-{year+1}"
    elif month in [3, 4, 5]:
        season = 'Summer'
        season_year = f"Summer_{year}"
    elif month in [6, 7, 8, 9]:
        season = 'Monsoon'
        season_year = f"Monsoon_{year}"
    else:  # Months 10, 11
        season = 'Post-Monsoon'
        season_year = f"Post-Monsoon_{year}"

    return pd.Series([season, season_year])

df_valid[['Season', 'Season_Year']] = df_valid['Date (LT)'].apply(get_season_info)

# 5. Temporal Feature Extraction (Dropping Day and Year to avoid overfitting)
df_valid['Hour'] = df_valid['Date (LT)'].dt.hour
df_valid['Month'] = df_valid['Date (LT)'].dt.month
df_valid['DayOfWeek'] = df_valid['Date (LT)'].dt.dayofweek

# 6. Map Target Variable to 3 Macro Hazard Tiers
def map_to_hazard_tier(category):
    if category in ['Good', 'Moderate']:
        return 'Healthy'
    elif category in ['Unhealthy for Sensitive Groups', 'Unhealthy']:
        return 'Caution'
    else: # 'Very Unhealthy', 'Hazardous'
        return 'Hazardous'

df_valid['Hazard_Tier'] = df_valid['AQI Category'].apply(map_to_hazard_tier)

# 7. Engineer 24-Hour Lagged Features (Diurnal Persistence)
df_valid['AQI_lag24'] = df_valid['AQI'].shift(24)
df_valid['Raw_Conc_lag24'] = df_valid['Raw Conc.'].shift(24)

# Drop rows where lag features could not be computed (first 24 hours)
df_clean = df_valid.dropna(subset=['AQI_lag24', 'Raw_Conc_lag24']).copy()

# 8. Save cleaned processed dataset to CSV
output_file = 'seasonaqi_processed_clean.csv'
df_clean.to_csv(output_file, index=False)

print("--- Preprocessing Complete ---")
print(f"Final Cleaned Dataset Shape: {df_clean.shape}")
print("\nClass Proportions (%):")
print(df_clean['Hazard_Tier'].value_counts(normalize=True) * 100)

--- Preprocessing Complete ---
Final Cleaned Dataset Shape: (50953, 35)

Class Proportions (%):
Hazard_Tier
Caution      54.459993
Healthy      26.932663
Hazardous    18.607344
Name: proportion, dtype: float64
